In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
import matplotlib.pyplot as plt
from textblob import TextBlob

In [ ]:
from textblob import TextBlob  # now it will work after installation


In [ ]:
years = [2006, 2007, 2008,2009,2010,2011,2012,2013,2014,2015]  # test first with 4 years

review_batches = []
for y in years:
    batch = session.sql(f"""
        SELECT *
        FROM stg_yelp_reviews
        WHERE YEAR(review_date) = {y}
    """).to_pandas()
    
    review_batches.append(batch)


In [ ]:
years = [2016,2017,2018,2019,2020,2021,2022]  # test first with 4 years

for y in years:
    batch = session.sql(f"""
        SELECT *
        FROM stg_yelp_reviews
        WHERE YEAR(review_date) = {y}
    """).to_pandas()
    
    review_batches.append(batch)


In [ ]:
df_reviews = pd.concat(review_batches, ignore_index=True)


In [ ]:
df_reviews.count()

In [ ]:
df_reviews.describe()

In [ ]:
df_reviews.info()

In [ ]:
df_reviews.shape

In [ ]:
## Missing values in the dataset

df_reviews.isnull().sum()

In [ ]:
## Duplicate records
df_reviews[df_reviews.duplicated()]

In [ ]:
df_reviews['FINAL_SENTIMENT'] = 'Neutral' 

In [ ]:
df_reviews.loc[
    (df_reviews['REVIEW_STARS'] >= 4) & (df_reviews['SENTIMENTS'] == 'Positive'),
    'FINAL_SENTIMENT'
] = 'Strong Positive'

df_reviews.loc[
    (df_reviews['REVIEW_STARS'] >= 4) & (df_reviews['SENTIMENTS'] == 'Negative'),
    'FINAL_SENTIMENT'
] = 'Mismatch Risk'

df_reviews.loc[
    (df_reviews['REVIEW_STARS'] <= 2) & (df_reviews['SENTIMENTS'] == 'Negative'),
    'FINAL_SENTIMENT'
] = 'Strong Negative'

df_reviews.loc[
    (df_reviews['REVIEW_STARS'] <= 2) & (df_reviews['SENTIMENTS'] == 'Positive'),
    'FINAL_SENTIMENT'
] = 'Anomaly'

In [ ]:
df_reviews['FINAL_SENTIMENT'].unique()

In [ ]:
df_reviews['FINAL_SENTIMENT'].value_counts()

In [ ]:
(df_reviews['FINAL_SENTIMENT'].value_counts()/len(df_reviews))*100


In [ ]:
df_reviews['FINAL_SENTIMENT'].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Sentiment_Final")
plt.ylabel("Count")
plt.title("Final Sentiment Distribution")
plt.show()

In [ ]:
(df_reviews['REVIEW_STARS'].value_counts()/len(df_reviews))*100

In [ ]:
import seaborn as sns

In [ ]:
df_reviews["REVIEW_STARS"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Stars")
plt.ylabel("Count")
plt.title("Review Star Distribution")
plt.show()

In [ ]:
(df_reviews['SENTIMENTS'].value_counts()/len(df_reviews))*100

In [ ]:
df_reviews["SENTIMENTS"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Sentiments")
plt.ylabel("Count")
plt.title("Sentiments Distribution")
plt.show()

In [ ]:
def sentiment_analyzer(text):
    if text is None:
        return 0
    blob = TextBlob(text)
    return blob.sentiment.polarity

In [ ]:
select count(*) from stg_yelp_reviews where sentiment_score!=0

In [ ]:
select FINAL_SENTIMENT from stg_yelp_reviews where sentiment_score is null 

In [ ]:
df_sentiment = session.sql("SELECT review_id,sentiment_score FROM stg_yelp_reviews").to_pandas()

In [ ]:
df_sentiment.count()

In [ ]:
df_reviews = df_reviews.merge(df_sentiment, on="REVIEW_ID", how="left")


In [ ]:
df_reviews.head()

In [ ]:
df_reviews["SENTIMENT_SCORE_x"].plot(kind="hist")
plt.title("Sentiment Score Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import scipy.stats as stats


In [ ]:
scores = df_reviews["SENTIMENT_SCORE_x"]

plt.figure()
stats.probplot(scores, dist="norm", plot=plt)
plt.title("Q-Q Plot for Sentiment Score")
plt.show()

In [ ]:
SELECT 
  AVG(sentiment_score) AS mean,
  STDDEV(sentiment_score) AS std,
  SKEW(sentiment_score) AS skew,
  KURTOSIS(sentiment_score) AS kurtosis
FROM stg_yelp_reviews
WHERE sentiment_score IS NOT NULL;

In [ ]:
df_reviews[['REVIEW_STARS','SENTIMENT_SCORE']].corr().iloc[0,1]


In [ ]:
# Correlation between sentiment polarity and review length (emotional intensity proxy)
corr_len_sent = df_reviews[['SENTIMENT_SCORE']].assign(
    REVIEW_LEN=df_reviews['REVIEW_GIVEN'].str.len()
).corr().iloc[0,1]

In [ ]:
corr_len_sent

In [ ]:
corr_star_len = df_reviews[['REVIEW_STARS']].assign(
    REVIEW_LEN=df_reviews['REVIEW_GIVEN'].str.len()
).corr().iloc[0,1]


In [ ]:
corr_star_len

In [ ]:
df_pos = df_reviews[df_reviews['SENTIMENTS'] == 'Positive']
df_neg = df_reviews[df_reviews['SENTIMENTS'] == 'Negative']
df_neu = df_reviews[df_reviews['SENTIMENTS'] == 'Neutral']

print(df_pos[['REVIEW_STARS','SENTIMENT_SCORE','REVIEW_LENGTH']].describe())
print(df_neg[['REVIEW_STARS','SENTIMENT_SCORE','REVIEW_LENGTH']].describe())
print(df_neu[['REVIEW_STARS','SENTIMENT_SCORE','REVIEW_LENGTH']].describe())

In [ ]:
import pandas as pd

# Make sure review length is numeric
df_reviews['REVIEW_LENGTH'] = df_reviews['REVIEW_GIVEN'].str.len()

# Compute correlations
corr_matrix = df_reviews[['REVIEW_STARS', 'SENTIMENT_SCORE', 'REVIEW_LENGTH']].corr()

print(corr_matrix)

In [ ]:
import seaborn as sns
plt.figure(figsize=(10,6))
sns.heatmap(corr_matrix,annot=True)